Overview — What We're Building
A tool where you:

Paste any song lyrics
It analyses the mood — happy, sad, angry, romantic etc.
Shows you the emotion breakdown as a nice chart
Gives you a summary of what the song is about emotionally

In [1]:
### installing libraries
# transformers gives us access to pre-trained AI models for free
# torch is the engine that runs them
!pip install transformers torch

In [3]:
### loading emotion detection model
from transformers import pipeline ### pre-trained model
emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k= None ### returns all emotions not just the top one
)
print("Model loaded")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded


In [4]:
  ### analysing lyrics

In [20]:
lyrics = """
My love was as cruel as the cities I lived in
Everyone looked worse in the light
There are so many lines that I've crossed unforgiven
I'll tell you the truth, but never goodbye
"""
### split lyrics into lines

lines = [line.strip() for line in lyrics.strip().split("\n")if line.strip()]


## analyse each line

results = []
for line in lines:
  emotions = emotion_classifier(line)[0]
  for emotion in emotions:
    results.append({
        "line":line,
        "emotion":emotion["label"],
        "score": round(emotion["score"],3)
    })

import pandas as pd
df = pd.DataFrame(results)
print(df.head(20))

                                                 line   emotion  score
0       My love was as cruel as the cities I lived in     anger  0.677
1       My love was as cruel as the cities I lived in   sadness  0.164
2       My love was as cruel as the cities I lived in   disgust  0.141
3       My love was as cruel as the cities I lived in      fear  0.009
4       My love was as cruel as the cities I lived in   neutral  0.004
5       My love was as cruel as the cities I lived in       joy  0.002
6       My love was as cruel as the cities I lived in  surprise  0.002
7                  Everyone looked worse in the light   disgust  0.953
8                  Everyone looked worse in the light   sadness  0.016
9                  Everyone looked worse in the light   neutral  0.013
10                 Everyone looked worse in the light      fear  0.010
11                 Everyone looked worse in the light     anger  0.007
12                 Everyone looked worse in the light  surprise  0.001
13    

In [21]:
### visualising the overall mood

import plotly.express as px

### average scores for each emotion- gives overall emotional profile

overall_mood = df.groupby("emotion")["score"].mean().reset_index()

## sorting from highest to lowest
overall_mood = overall_mood.sort_values("score", ascending=False)

 ##Define colours for each emotion
colour_map = {
    "joy": "#FFD700",
    "sadness": "#6495ED",
    "anger": "#FF4500",
    "fear": "#9370DB",
    "surprise": "#FF69B4",
    "disgust": "#32CD32",
    "neutral": "#A9A9A9"
}

## creating a bar chart
fig = px.bar(
    overall_mood,
    x="emotion",
    y="score",
    color="emotion",
    color_discrete_map=colour_map,
    title="Emotional Profile of Your Song",
    labels={"score": "Average Intensity", "emotion": "Emotion"},
)

fig.show()

In [22]:
# Find the dominant emotion (highest average score)
top_emotion = overall_mood.iloc[0]["emotion"]
top_score = overall_mood.iloc[0]["score"]

# Find the strongest non-neutral emotion
non_neutral = overall_mood[overall_mood["emotion"] != "neutral"]
top_feeling = non_neutral.iloc[0]["emotion"]

# Emoji map for fun!
emoji_map = {
    "joy": "😊", "sadness": "😔", "anger": "😠",
    "fear": "😨", "disgust": "🤢", "surprise": "😲", "neutral": "😐"
}

print("=" * 45)
print("        🎵 SONG MOOD ANALYSIS REPORT 🎵")
print("=" * 45)
print(f"Overall tone:      {emoji_map[top_emotion]}  {top_emotion.upper()} ({top_score:.0%})")
print(f"Strongest feeling: {emoji_map[top_feeling]}  {top_feeling.upper()}")
print("-" * 45)
print("Full breakdown:")
for _, row in overall_mood.iterrows():
    bar = "█" * int(row["score"] * 20)
    print(f"  {emoji_map[row['emotion']]} {row['emotion']:<10} {bar} {row['score']:.0%}")
print("=" * 45)

        🎵 SONG MOOD ANALYSIS REPORT 🎵
Overall tone:      🤢  DISGUST (33%)
Strongest feeling: 🤢  DISGUST
---------------------------------------------
Full breakdown:
  🤢 disgust    ██████ 33%
  😠 anger      ██████ 31%
  😔 sadness    ███ 18%
  😐 neutral    ███ 15%
  😨 fear        2%
  😊 joy         0%
  😲 surprise    0%


In [23]:
import plotly.graph_objects as go
import numpy as np

# Get unique emotions and lines
emotions = ["joy", "sadness", "anger", "fear", "surprise", "disgust", "neutral"]
lines = df["line"].unique().tolist()

# Build a matrix of scores — rows = emotions, columns = lines
# Think of it like a grid where each cell = how strongly that emotion appears in that line
matrix = []
for emotion in emotions:
    row = []
    for line in lines:
        score = df[(df["line"] == line) & (df["emotion"] == emotion)]["score"].values
        row.append(float(score[0]) if len(score) > 0 else 0.0)
    matrix.append(row)

# Shorten long lines so they fit nicely on the chart
short_lines = [l[:30] + "..." if len(l) > 30 else l for l in lines]

# Create the heatmap
fig = go.Figure(data=go.Heatmap(
    z=matrix,
    x=short_lines,       # Lines of lyrics along the bottom
    y=emotions,          # Emotions along the side
    colorscale="RdYlGn", # Red = high intensity, Green = low
    hoverongaps=False
))

fig.update_layout(
    title="🎵 Emotion Journey Through the Song",
    xaxis_title="Lyrics (line by line)",
    yaxis_title="Emotion",
    xaxis_tickangle=-45,  # Tilt the labels so they don't overlap
    height=500
)

fig.show()